# Pneumonia CNN comparison

## Import libraries, define common methods and mount the storage

In [ ]:
import gc
import os
import time
from google.colab import drive
from PIL import Image
import numpy as np
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import InceptionV3, VGG19, ResNet50
from tensorflow.keras.applications.inception_v3 import preprocess_input as InceptionPreprocess
from tensorflow.keras.applications.vgg19 import preprocess_input as VGGPreprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as ResNetPreprocess
from tensorflow.keras.layers import (
    Input, Dense, Lambda, GlobalAveragePooling2D, Dropout, BatchNormalization,
    RandomFlip, RandomRotation, RandomTranslation, RandomCrop, Resizing
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt


def print_time(started: float):
  elapsed = time.time() - started
  print(f'Minutes: {elapsed // 60}, seconds: {elapsed % 60}')

MOUNT_POINT = '/content/drive'

if not os.path.ismount(MOUNT_POINT):
  drive.mount(MOUNT_POINT, readonly=True)

## Preprocessing

### Constants, variables, classes

In [ ]:
ROOT_PATH = f'{MOUNT_POINT}/MyDrive/Pneumonia CNN comparison'
INCEPTION_ROOT_PATH = f'{ROOT_PATH}/Inception'
VGG_RESNET_ROOT_PATH = f'{ROOT_PATH}/VGG_ResNet'
INCEPTION = 'Inception-V3'; VGG = 'VGG-19'; RESNET = 'ResNet-50'
NORMAL = 'Normal'; NORMAL_NUM = 0
PNEUMONIA = 'Pneumonia'; PNEUMONIA_NUM = 1
RANDOM_STATE = 10

class CnnModel:
  model_name: str
  image_size: int
  image_channels: int
  max_images: int
  images_per_class: int
  use_augmentation: bool
  augmentation_images: int
  aug_per_class: int

  root_path: str
  training_path: str
  validation_path: str
  testing_path: str

  def __init__(self, root_path: str, model_name: str, image_size: int, image_channels: int,
               max_images: int, use_augmentation: bool, augmentation_images: int = 0) -> None:
    self.model_name = model_name
    self.image_size = image_size
    self.image_channels = image_channels
    self.use_augmentation = use_augmentation

    self.max_images = max_images
    self.images_per_class = max_images // 2
    self.augmentation_images = augmentation_images
    self.aug_per_class = augmentation_images // 2

    self.root_path = root_path
    self.training_path = f'{root_path}/Training'
    self.validation_path = f'{root_path}/Validation'
    self.testing_path = f'{root_path}/Testing'

### Model selection

In [ ]:
cnn_model = CnnModel(VGG_RESNET_ROOT_PATH, RESNET, 224, 3, 3700, False, 0)

### Load images

In [ ]:
def load_images(folder_path: str, take: int = 1_000_000):
  file_names = sorted(os.listdir(folder_path))
  file_names = shuffle(file_names, random_state=RANDOM_STATE)[:take]

  datas = []
  started = time.time()
  for index, file_name in enumerate(file_names):
    image = Image.open(f'{folder_path}/{file_name}').convert('RGB')
    arr = np.asarray(image, dtype=np.uint8)
    datas.append(arr)

    if (index + 1) % 500 == 0:
      print(f'Loaded {index + 1} images')

  datas = np.array(datas, dtype=np.uint8)

  print(f'Shape: {datas.shape}')
  print_time(started)

  return datas

print('Loading training images')
training_normal = load_images(cnn_model.training_path + '/' + NORMAL, cnn_model.images_per_class)
training_pneumonia = load_images(cnn_model.training_path + '/' + PNEUMONIA, cnn_model.images_per_class)

print('Loading validation images')
validation_normal = load_images(cnn_model.validation_path + '/' + NORMAL)
validation_pneumonia = load_images(cnn_model.validation_path + '/' + PNEUMONIA)

print('Loading testing images')
testing_normal = load_images(cnn_model.testing_path + '/' + NORMAL)
testing_pneumonia = load_images(cnn_model.testing_path + '/' + PNEUMONIA)

### Combine and convert to X and Y

In [ ]:
def combine(normal_images: np.ndarray, pneumonia_images: np.ndarray):
  normal_y = np.full(len(normal_images), NORMAL_NUM, dtype=np.float32)
  pneumonia_y = np.full(len(pneumonia_images), PNEUMONIA_NUM, dtype=np.float32)

  combined_images = np.concatenate((normal_images, pneumonia_images), axis=0)
  combined_ys = np.concatenate((normal_y, pneumonia_y), axis=0)

  return shuffle(combined_images, combined_ys, random_state=RANDOM_STATE)

x_train, y_train = combine(training_normal, training_pneumonia)
x_validation, y_validation = combine(validation_normal, validation_pneumonia)
x_test, y_test = combine(testing_normal, testing_pneumonia)

del training_normal, training_pneumonia, validation_normal, validation_pneumonia, testing_normal, testing_pneumonia
gc.collect()

### Augmentation

In [ ]:
def augment(x: np.ndarray, y: np.ndarray, aug_per_class: int, batch_size: int = 32):
  if aug_per_class <= 0:
    return x, y

  image_size = x.shape[1]
  crop_size = int(image_size * 0.90)

  aug_pipeline = keras.Sequential([
      RandomFlip('horizontal', seed=RANDOM_STATE),
      RandomRotation(0.04, fill_mode='nearest', seed=RANDOM_STATE + 1),
      RandomTranslation(0.05, 0.05, fill_mode='nearest', seed=RANDOM_STATE + 2),
      RandomCrop(crop_size, crop_size, seed=RANDOM_STATE + 3),
      Resizing(image_size, image_size),
  ], name='offline_augmentation')

  rng = np.random.default_rng(RANDOM_STATE)
  generated_x = [x]
  generated_y = [y]

  for class_label in (NORMAL_NUM, PNEUMONIA_NUM):
    class_indices = np.where(y == class_label)[0]
    chosen = rng.choice(class_indices, size=aug_per_class, replace=True)

    for start in range(0, aug_per_class, batch_size):
      batch_uint8 = x[chosen[start:start + batch_size]]
      batch_float = tf.cast(batch_uint8, tf.float32)
      aug_batch = aug_pipeline(batch_float, training=True).numpy()
      aug_uint8 = np.clip(np.round(aug_batch), 0, 255).astype(np.uint8)
      generated_x.append(aug_uint8)

    generated_y.append(np.full(aug_per_class, class_label, dtype=np.float32))

  x_all = np.concatenate(generated_x, axis=0)
  y_all = np.concatenate(generated_y, axis=0)

  return shuffle(x_all, y_all, random_state=RANDOM_STATE)

if cnn_model.use_augmentation and cnn_model.aug_per_class > 0:
  aug_start = time.time()
  x_train, y_train = augment(x_train, y_train, cnn_model.aug_per_class)
  print_time(aug_start)

print(f'Train: {x_train.shape} (Normal: {int(np.sum(y_train == 0))}, Pneumonia: {int(np.sum(y_train == 1))})')
print(f'Val:   {x_validation.shape}, Test: {x_test.shape}')

## Training

### Model selection

In [ ]:
keras.utils.set_random_seed(RANDOM_STATE)

base_preprocess = None
base_model = None

if cnn_model.model_name == INCEPTION:
  base_model = InceptionV3(
      weights='imagenet',
      include_top=False,
      input_shape=(cnn_model.image_size, cnn_model.image_size, cnn_model.image_channels))
  base_preprocess = InceptionPreprocess
elif cnn_model.model_name == VGG:
  base_model = VGG19(
      weights='imagenet',
      include_top=False,
      input_shape=(cnn_model.image_size, cnn_model.image_size, cnn_model.image_channels))
  base_preprocess = VGGPreprocess
else:
  base_model = ResNet50(
      weights='imagenet',
      include_top=False,
      input_shape=(cnn_model.image_size, cnn_model.image_size, cnn_model.image_channels))
  base_preprocess = ResNetPreprocess

base_model.trainable = False

### Neural network building

In [ ]:
model = keras.Sequential([
    Input(shape=(cnn_model.image_size, cnn_model.image_size, cnn_model.image_channels)),
    Lambda(
        lambda x: base_preprocess(tf.cast(x, tf.float32)),
        output_shape=(cnn_model.image_size, cnn_model.image_size, cnn_model.image_channels)
    ),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

print(model.summary())

### Callbacks

In [ ]:
def make_callbacks(patience: int = 10):
  return [
      EarlyStopping(
          monitor='val_loss',
          patience=patience,
          restore_best_weights=True,
          verbose=1
      ),
      ReduceLROnPlateau(
          monitor='val_loss',
          factor=0.2,
          patience=4,
          min_lr=1e-7,
          verbose=1
      )
  ]

### Compile and Train

In [ ]:
training_start_time = time.time()

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

training_stage = model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=50,
    validation_data=(x_validation, y_validation),
    callbacks=make_callbacks(patience=10)
)

history = training_stage.history

print_time(training_start_time)

## Evaluation

### Evaluate

In [ ]:
evaluation = model.evaluate(x_test, y_test)

### Confusion matrix

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

y_true = y_test.astype(int)
y_prob = model.predict(x_test, batch_size=32).ravel()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)

aug_count = cnn_model.augmentation_images if cnn_model.use_augmentation else 0
run_title = f'{cnn_model.model_name} ({cnn_model.max_images} org + {aug_count} aug)'

print(f'=== {run_title} ===')
print(f'Accuracy:    {accuracy:.4f}')
print(f'Precision:   {precision:.4f}')
print(f'Recall:      {recall:.4f}  (Sensitivity / Jautrumas)')
print(f'Specificity: {specificity:.4f}  (Specifiškumas)')
print(f'F1-score:    {f1:.4f}')
print(f'ROC-AUC:     {roc_auc:.4f}\n')
print(classification_report(y_true, y_pred, target_names=[NORMAL, PNEUMONIA], digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[NORMAL, PNEUMONIA])
disp.plot(ax=axes[0], cmap='Blues', values_format='d', colorbar=False)
axes[0].set_title(f'Klaidų matrica\n{run_title}')
axes[0].set_xlabel('Prognozuota klasė')
axes[0].set_ylabel('Tikroji klasė')

fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Atsitiktinis (AUC = 0.50)')
axes[1].set_title(f'ROC kreivė\n{run_title}')
axes[1].set_xlabel('False Positive Rate (1 - Specifiškumas)')
axes[1].set_ylabel('True Positive Rate (Jautrumas)')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Graphs

#### Model accuracy

In [ ]:
plt.plot(history['accuracy'], label='Mokymo')
plt.plot(history['val_accuracy'], label='Validavimo')
plt.xlabel('Epochos')
plt.ylabel('Tikslumas')
plt.legend(loc='upper left')
plt.title('Modelio tikslumas')
plt.show()

#### Model loss

In [ ]:
plt.plot(history['loss'], label='Mokymo')
plt.plot(history['val_loss'], label='Validavimo')
plt.xlabel('Epochos')
plt.ylabel('Loss')
plt.legend(loc='upper left')
plt.title('Modelio loss')
plt.show()